In [ ]:
# step4_keyword_validation_final.py

import os
import requests
import pandas as pd
from dotenv import load_dotenv
from time import sleep
import base64

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 6)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("No GitHub tokens found in All_Tokens.env")

token_index = 0
def get_headers():
    global token_index
    token = tokens[token_index]
    token_index = (token_index + 1) % len(tokens)
    return {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "android-repo-crawler/1.0"
    }

# === File paths ===
input_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step3_android_detection_output.csv"
output_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step4_keyword_check_output.csv"

# === Read data ===
df = pd.read_csv(input_path)

# === Prepare updated fields ===
updated_name = []
updated_description = []
updated_topics = []
updated_readme_flags = []
updated_android_keywords = []
keyword_checks = []

# === Loop through repos ===
for i, row in df.iterrows():
    if row["status"] != "pass":
        # Keep unchanged for rejected repos
        updated_name.append(row["name"])
        updated_description.append(row["description"])
        updated_topics.append(row["topics"])
        updated_readme_flags.append(row["android_in_readme"])
        updated_android_keywords.append(row["android_keyword"])
        keyword_checks.append("reject")
        continue

    repo = row["full_name"]

    # === Step 1: Real-time metadata ===
    name = row.get("name", "")
    description = row.get("description", "")
    topics_list = str(row.get("topics", "")).lower().split(",")
    in_readme = False

    repo_url = f"https://api.github.com/repos/{repo}"
    r = requests.get(repo_url, headers=get_headers())
    if r.status_code == 200:
        metadata = r.json()
        name = str(metadata.get("name", "")).lower()
        description = str(metadata.get("description", "")).lower()
    updated_name.append(name)
    updated_description.append(description)

    # === Step 2: Real-time topics ===
    topics_url = f"https://api.github.com/repos/{repo}/topics"
    r_topics = requests.get(topics_url, headers=get_headers())
    if r_topics.status_code == 200:
        topic_names = r_topics.json().get("names", [])
        updated_topics.append(",".join(topic_names))
        topics_list = [t.lower() for t in topic_names]
    else:
        updated_topics.append(row["topics"])

    # === Step 3: Real-time README ===
    readme_url = f"https://api.github.com/repos/{repo}/readme"
    r_readme = requests.get(readme_url, headers=get_headers())
    if r_readme.status_code == 200:
        content = r_readme.json().get("content", "")
        try:
            decoded = base64.b64decode(content).decode("utf-8", errors="ignore").lower()
            if "android" in decoded:
                in_readme = True
        except Exception as e:
            pass
    updated_readme_flags.append("yes" if in_readme else "no")

    # === Step 4: Final keyword decision ===
    found = (
        "android" in name or
        "android" in description or
        any("android" in t for t in topics_list) or
        in_readme
    )

    updated_android_keywords.append("yes" if found else "no")
    keyword_checks.append("pass" if found else "reject")

    if i % 100 == 0:
        print(f"🔎 Checked {i+1} repos...")

# === Save final output ===
df["name"] = updated_name
df["description"] = updated_description
df["topics"] = updated_topics
df["android_in_readme"] = updated_readme_flags
df["android_keyword"] = updated_android_keywords
df["keyword_check"] = keyword_checks

df.to_csv(output_path, index=False)
print(f"✅ Step 4 complete. Saved to: {output_path}")
